In [1]:
# For processing the timeseries
import pandas as pd, os, datetime
import numpy as np

# For plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
from scipy.stats import gaussian_kde
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
def jitter(group):
    duplicates = group.groupby(['lat', 'lon']).cumcount()
    
    # Jitter function: add a small random offset
    np.random.seed(42)  # For reproducibility
    jitter_strength = 0.01  # Adjust as needed; degrees latitude/longitude
    
    group['lat_jittered'] = group['lat'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates
    group['lon_jittered'] = group['lon'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates

    return group

In [4]:
gen_details = pd.read_csv(f"{nmap_path}/gen_info.csv")

In [5]:
# A temporary patch to fix fuel types
gen_details['fuel_source_primary'] = gen_details['fuel_source_primary'].replace({'Solar - Solar': 'Solar','Wind - Wind': 'Wind'})
gen_details = jitter(gen_details)

In [ ]:
hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv")

In [ ]:
def process_group(grp, gen_fpath, hw_tseries, start_date=None, end_date=None):
    """
    Processes a single group of generator data.

    Parameters:
        grp (pd.DataFrame): A single group from gen_details (e.g., from groupby('region')).
        gen_fpath (str): Path to directory containing CSV files named by DUID (e.g., DUID.csv).
        hw_tseries (pd.DataFrame): DataFrame with columns 'time', 'DUID', and data to merge.
        start_date (str): Start date for subsetting the time series.
        end_date (str): End date for subsetting the time series.

    Returns:
        pd.DataFrame: The merged result for the group.
    """
    safe_duids = [duid.replace("/", "_").replace("\\", "_") for duid in grp['DUID']]
    gen_locs = [f"{gen_fpath}/{duid}.csv" for duid in safe_duids]
    dfs = [pd.read_csv(fp,dtype='object') for fp in gen_locs if os.path.exists(fp)]
    print(f"Loaded {len(dfs)} CSV file(s) out of {len(gen_locs)} expected.")

    if not dfs:
        return None

    # This is in case of accidental mid-file headers
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # This is to correct the column types after removing header rows
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)
    
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    # Group by DUID and hourly time, sum TOTALMWh
    agg_func = {'TOTALMWh':'sum','TOTALCLEARED':'sum','AGCSTATUS':'max'}
    grouped = dfs.groupby(['DUID', pd.Grouper(freq='1d')]).agg(agg_func).reset_index()

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index(['time']).sort_index()
    hw_tseries = hw_tseries.loc[sdate:edate]
    hw_tseries = hw_tseries.reset_index().set_index(['DUID','time']).sort_index()

    merged = pd.merge_asof(
        grouped.sort_values(by=['time', 'DUID']),
        hw_tseries.sort_values(by=['time', 'DUID']),
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("1d"),
        direction='nearest'
    )

    merged = merged.dropna(how='all')

    return merged

In [ ]:
def select_group(gen_details=gen_details, state=None,ftype=None):
    if state is not None and ftype is not None:
        groups = gen_details.groupby(['region','fuel_source_primary'])
        grp = groups.get_group((state,ftype))
    elif state is not None:
        groups = gen_details.groupby('region')
        grp = groups.get_group(state)
    elif ftype is not None:
        groups = gen_details.groupby('fuel_source_primary')
        grp = groups.get_group(ftype)

    else:
        grp = gen_details

    return grp


In [ ]:
sdate, edate = '2018-11-01','2019-3-31'
# sdate, edate = '2016-06-01','2019-06-30'
group = select_group(state='NSW1').copy()
df = process_group(group, gen_fpath, hw_tseries,sdate,edate)

In [ ]:
demand = pd.read_csv('/scratch/ng72/ms5578/time_series/state_demand.csv')
demand['time'] = pd.to_datetime(demand['time'])
demand = demand.set_index('time').sort_index()
demand = demand.loc[sdate:edate]
demand = demand.loc[demand['REGIONID'] =='NSW1']
demand = demand.resample('D', level=0).sum()

In [ ]:
df = df.merge(gen_details[['DUID', 'reg_cap_mw','technology_type_primary','fuel_source_primary','lat_jittered','lon_jittered']], on='DUID', how='left')
df['reg_cap_mw'] = df['reg_cap_mw'].astype('float') * 24
df['cap_norm'] = df['TOTALMWh']/df['reg_cap_mw']
df['min_max'] = df.groupby('DUID')['TOTALMWh'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
df = df[~((df['TOTALMWh'] < 0) | (df['cap_norm'] <0))]
mask = (df['fuel_source_primary'].isin([
    'Water', 'Natural Gas Pipeline', 'Black Coal', 'Coal Seam Methane',
    'Brown Coal', 'Diesel', 'Kerosene'
])) & (df['AGCSTATUS'] == 0)

df.loc[mask, 'TOTALMWh'] = 0

In [ ]:
def highLights(df, fig, variable, level, mode, fillcolor, layer):
    """
    Set a specified color as background for given
    levels of a specified variable using a shape.
    
    Keyword arguments:
    ==================
    fig -- plotly figure
    variable -- column name in a pandas dataframe
    level -- int or float
    mode -- set threshold above or below
    fillcolor -- any color type that plotly can handle
    layer -- position of shape in plotly fiugre, like "below"
    
    """
    
    if mode == 'above':
        m = df[variable].gt(level)
    
    if mode == 'below':
        m = df[variable].lt(level)
        
    df1 = df[m].groupby((~m).cumsum())['time'].agg(['first','last'])

    for index, row in df1.iterrows():
        fig.add_shape(
            type="rect",
            xref="x",
            yref="paper",
            x0=row['first'],
            y0=0,
            x1=row['last'],
            y1=1,
            line=dict(color="rgba(0,0,0,0)", width=3),
            fillcolor="rgba(100,100,100,0.2)",
            layer=layer
        )
    return(fig)

In [ ]:
fig, ax = plt.subplots() 
ax.hist(df[~(df['cap_norm'] == 0)]['cap_norm'], bins=100, color='skyblue', edgecolor='black');
ax.set_title("Generation as a Fraction of Capacity")
ax.set_xlabel("Daily Generation as a Fraction of Daily Maximum Nameplate Capacity")
ax.set_ylabel("Frequency")

In [ ]:
# Step 1: Count non-heatwave (0) and heatwave (1) days per DUID
agc_df= df[~((df['fuel_source_primary'].isin(['Water', 'Natural Gas Pipeline', 'Black Coal'])) & (df['AGCSTATUS'] == 0))]
day_counts = agc_df.groupby(['DUID', 'EHF_flag']).size().unstack(fill_value=0).reset_index()
day_counts.columns.name = None
day_counts = day_counts.rename(columns={0: 'no_hw', 1: 'in_hw'})

# Step 2: Merge with fuel source info
day_counts = day_counts.merge(group[['DUID', 'fuel_source_primary', 'station_name']], on='DUID')

# Step 3: Sort for visual grouping
day_counts = day_counts.sort_values(by=['fuel_source_primary', 'DUID'])

# Step 4: Assign consistent color per fuel type
fuel_types = day_counts['fuel_source_primary'].unique()
color_palette = px.colors.qualitative.Set2
color_map = {fuel: color_palette[i % len(color_palette)] for i, fuel in enumerate(fuel_types)}
day_counts['color'] = day_counts['fuel_source_primary'].map(color_map)

# Step 5: Create stacked bars
fig = go.Figure()

# --- Non-heatwave bars (solid color) ---
fig.add_trace(go.Bar(
    x=day_counts['DUID'],
    y=day_counts['no_hw'],
    name='Not in Heatwave',
    marker_color=day_counts['color'],
    customdata=day_counts[['fuel_source_primary', 'station_name']].values,
    hovertemplate=(
        '<b>DUID:</b> %{x}<br>' +
        '<b>Name:</b> %{customdata[1]}<br>' +
        '<b>Days (Not in HW):</b> %{y}<br>' +
        '<b>Fuel:</b> %{customdata[0]}<extra></extra>'
    )
))

# --- Heatwave bars (same color, with hatching) ---
fig.add_trace(go.Bar(
    x=day_counts['DUID'],
    y=day_counts['in_hw'],
    name='In Heatwave',
    marker=dict(
        color='#798b94',
        pattern=dict(shape='x', solidity=0.7)
    ),
    customdata=day_counts[['fuel_source_primary', 'station_name']].values,
    hovertemplate=(
        '<b>DUID:</b> %{x}<br>' +
        '<b>Name:</b> %{customdata[1]}<br>' +
        '<b>Days (In HW):</b> %{y}<br>' +
        '<b>Fuel:</b> %{customdata[0]}<extra></extra>'
    )
))
# Step 6: Layout settings
fig.update_layout(
    title='Heatwave vs Non-Heatwave Days by Generator (Colored by Fuel Type, Hatching in HW)',
    xaxis_title='Generator (DUID)',
    yaxis_title='Number of Days',
    barmode='stack',
    height=600,
    xaxis=dict(
        tickmode='array',
        tickvals=day_counts['DUID'],
        ticktext=day_counts['DUID'],
        tickangle=90
    ),
    legend=dict(
        title='Legend',
        traceorder='normal'
    )
)

fig.show()


In [ ]:
# --- 1. Group data ---
# % of generators in/out of heatwave
gens_in_hw = df.groupby('time')['EHF_flag'].value_counts().unstack(fill_value=0)
gens_percent = gens_in_hw.div(gens_in_hw.sum(axis=1), axis=0) * 100

# Total generation split by EHF_flag
gen_split = df.groupby(['time', 'EHF_flag'])['TOTALMWh'].sum().unstack(fill_value=0)

import plotly.graph_objects as go

# --- Heatwave Participation (% stacked bars) ---
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=gens_percent.index.astype(str),
    y=gens_percent[0],
    mode='lines',
    name='Not in Heatwave (%)',
    line=dict(color='lightblue'),
    stackgroup='one'
))

fig1.add_trace(go.Scatter(
    x=gens_percent.index.astype(str),
    y=gens_percent[1],
    mode='lines',
    name='In Heatwave (%)',
    line=dict(color='tomato'),
    stackgroup='one'
))

fig1.update_layout(
    title='Percentage of Generators in Heatwave per Day',
    xaxis_title='Date',
    yaxis_title='Participation (%)',
    yaxis=dict(ticksuffix='%', range=[0, 100]),
    height=400,
    legend=dict(title='EHF_flag'),
    hovermode='x unified'
)

fig1.show()

In [ ]:
df_mask_ehf = df.copy()
df_mask_ehf.loc[df_mask_ehf['EHF_flag'] == 0, 'EHF_val'] = 0

In [ ]:
# # Create base figure with line chart
# fig = px.line(
#     df_mask_ehf,
#     x='time',
#     y='EHF_val',
#     color='DUID',
#     labels={
#         'time': 'Date',
#         'Clear Gap': 'Clear Gap (MWh)',
#         'fuel_source_primary': 'Fuel Source'
#     },
#     title='EHF by DUID'
# )

# fig.show()

In [ ]:
def compute_conditional_cumulative_avg(df, value_col='TOTALMWh', fuel_col='fuel_source_primary',
                                       duid_col='DUID', ehf_col='EHF_flag', agc_col='AGCSTATUS',
                                       fuels_requiring_agc=None):
    """
    Compute cumulative average of a value column, excluding rows based on EHF_flag,
    and optionally AGCSTATUS for certain fuel types.
    """
    
    if fuels_requiring_agc is None:
        fuels_requiring_agc = ['Water', 'Natural Gas Pipeline', 'Black Coal']

    df = df.copy()
    df[ehf_col] = df[ehf_col].astype(bool)

    # Compute exclusion mask
    df['exclude_flag'] = np.where(
        df[fuel_col].isin(fuels_requiring_agc),
        df[ehf_col] | (df[agc_col] != 1),
        df[ehf_col]
    )

    # Define cumulative average function
    def cumulative_avg_with_exclusion(sub_df):
        values = sub_df[value_col]
        exclude_flags = sub_df['exclude_flag']
        result = []
        running_sum = 0.0
        count = 0
        for v, excl in zip(values, exclude_flags):
            if not excl and not pd.isna(v):
                running_sum += v
                count += 1
            result.append(running_sum / count if count > 0 else np.nan)
        return pd.Series(result, index=sub_df.index)

    # Group and apply without touching grouping columns
    results = []
    for duid, sub_df in df.groupby(duid_col):
        result = cumulative_avg_with_exclusion(sub_df)
        results.append(result)

    # Combine all into a single Series (aligned to df index)
    grouped_cum_avg = pd.concat(results).sort_index()

    return grouped_cum_avg


In [ ]:
def compute_conditional_weighted_cumulative_avg(
    df, value_col='TOTALMWh', fuel_col='fuel_source_primary',
    duid_col='DUID', ehf_col='EHF_flag', agc_col='AGCSTATUS',
    fuels_requiring_agc=None
):
    """
    Compute weighted cumulative average of a value column, excluding rows based on EHF_flag,
    and optionally AGCSTATUS for certain fuel types. The most recent row is weighted highest.
    """
    if fuels_requiring_agc is None:
        fuels_requiring_agc = ['Water', 'Natural Gas Pipeline', 'Black Coal']

    df = df.copy()
    df[ehf_col] = df[ehf_col].astype(bool)

    # Compute exclusion mask
    df['exclude_flag'] = np.where(
        df[fuel_col].isin(fuels_requiring_agc),
        df[ehf_col] | (df[agc_col] != 1),
        df[ehf_col]
    )

    def weighted_cumulative_avg_with_exclusion(sub_df):
        values = sub_df[value_col]
        exclude_flags = sub_df['exclude_flag']
        result = []
        weighted_sum = 0.0
        total_weight = 0.0
        weight = 0
        for v, excl in zip(values, exclude_flags):
            weight += 1
            if not excl and not pd.isna(v):
                weighted_sum += v * weight
                total_weight += weight
            result.append(weighted_sum / total_weight if total_weight > 0 else np.nan)
        return pd.Series(result, index=sub_df.index)

    # Group and apply without touching grouping columns
    results = []
    for duid, sub_df in df.groupby(duid_col):
        result = weighted_cumulative_avg_with_exclusion(sub_df)
        results.append(result)

    # Combine all into a single Series (aligned to df index)
    grouped_weighted_cum_avg = pd.concat(results).sort_index()

    return grouped_weighted_cum_avg

In [ ]:
df['grouped_cum_avg'] = compute_conditional_cumulative_avg(
    df,
    value_col='TOTALMWh',
    fuels_requiring_agc=['Water', 'Black Coal','Natural Gas Pipeline']
)
cum_avg_df = df.copy()

df = df.set_index('time').loc['2018-12-01':'2019-03-31'].reset_index()

In [ ]:
fig = go.Figure()

# Group by fuel type
fuel_groups = cum_avg_df.groupby('fuel_source_primary')

trace_visibility = []
all_traces = []

# Add one trace per DUID grouped by fuel type
for fuel, group in fuel_groups:
    group_by_duid = group.groupby('DUID')
    visible_traces = []
    for duid, duid_df in group_by_duid:
        trace = go.Scatter(
            x=duid_df['time'],
            y=duid_df['grouped_cum_avg'],
            name=duid,
            mode='lines',
            hovertext=duid_df['fuel_source_primary'],
            visible=True  # Default all traces visible initially
        )
        fig.add_trace(trace)
        all_traces.append(duid)
        visible_traces.append(1)
    trace_visibility.append(visible_traces)

# Build visibility masks
full_visibility = []
start = 0
for vis in trace_visibility:
    mask = [False] * len(fig.data)
    for i in range(len(vis)):
        mask[start + i] = True
    full_visibility.append(mask)
    start += len(vis)

# All-visible mask
all_visible = [True] * len(fig.data)

buttons = [
    dict(
        label="All",
        method="update",
        args=[
            {"visible": all_visible},
            {"title": {"text": "Cumulative average by DUID"}}
        ]
    )
]

for fuel, vis in zip(fuel_groups.groups.keys(), full_visibility):
    buttons.append(
        dict(
            label=fuel,
            method="update",
            args=[
                {"visible": vis},
                {"title": {"text": f"Cumulative average by DUID (Fuel: {fuel})"}}
            ]
        )
    )

# Add shaded date range
fig.add_vrect(
    x0="2018-12-27",
    x1="2019-02-10",
    fillcolor="red",
    opacity=0.2,
    layer="below",
    line_width=0
)

# Final layout
fig.update_layout(
    title="Cumulative average by DUID",
    xaxis_title="Date",
    yaxis_title="Cumulative Average",
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            direction="down",
            x=1.05,
            xanchor="left",
            y=1.1,
            yanchor="top"
        )
    ]
)

fig.show()


In [ ]:
df['dev_from_avg'] = df['TOTALMWh'] - df['grouped_cum_avg']
df['dev_norm'] = df.groupby('DUID')['dev_from_avg'].transform(lambda x: (2*(x - x.min()) / (x.max() - x.min()))-1)
df = df[~((df['fuel_source_primary'].isin(['Water', 'Natural Gas Pipeline', 'Black Coal'])) & (df['AGCSTATUS'] == 0))]

In [ ]:
import plotly.graph_objects as go

# Create figure manually
fig = go.Figure()

# Group by fuel type
fuel_groups = df.groupby('fuel_source_primary')

trace_visibility = []
all_traces = []

# Add one trace per DUID grouped by fuel type
for fuel, group in fuel_groups:
    group_by_duid = group.groupby('DUID')
    visible_traces = []
    for duid, duid_df in group_by_duid:
        trace = go.Scatter(
            x=duid_df['time'],
            y=duid_df['dev_from_avg'],
            name=duid,
            mode='lines',
            hovertext=duid_df['fuel_source_primary'],
            visible=True  # Default all traces visible initially
        )
        fig.add_trace(trace)
        all_traces.append(duid)
        visible_traces.append(1)
    trace_visibility.append(visible_traces)

# Build visibility masks
full_visibility = []
start = 0
for vis in trace_visibility:
    mask = [False] * len(fig.data)
    for i in range(len(vis)):
        mask[start + i] = True
    full_visibility.append(mask)
    start += len(vis)

# All-visible mask
all_visible = [True] * len(fig.data)

# Build dropdown buttons
buttons = [
    dict(
        label="All",
        method="update",
        args=[
            {"visible": all_visible},
            {"title": {"text": "Difference from average by DUID"}}
        ]
    )
]

for fuel, vis in zip(fuel_groups.groups.keys(), full_visibility):
    buttons.append(
        dict(
            label=fuel,
            method="update",
            args=[
                {"visible": vis},
                {"title": {"text": f"Difference from average by DUID (Fuel: {fuel})"}}
            ]
        )
    )

# Add shaded date range
fig.add_vrect(
    x0="2018-12-27",
    x1="2019-02-10",
    fillcolor="red",
    opacity=0.2,
    layer="below",
    line_width=0
)

# Final layout
fig.update_layout(
    title="Difference from average by DUID",
    xaxis_title="Date",
    yaxis_title="Cumulative Average",
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            direction="down",
            x=1.05,
            xanchor="left",
            y=1.1,
            yanchor="top"
        )
    ]
)

fig.show()


In [ ]:
fig = px.bar(
    df,
    x='time',
    y='dev_from_avg',
    color='fuel_source_primary',
    title='Difference in Average Generation in MWh',
    hover_data=['DUID', 'time','TOTALMWh','EHF_flag'],
    labels={'dev_from_avg': 'Difference in Avg Generation',
           'EHF_flag':"In HW:"},
    height=500
)

# Add shaded red rectangle (vrect) from Dec 27 to Feb 10
fig.add_vrect(
    x0="2018-12-27",  # Adjust year as needed
    x1="2019-02-10",
    fillcolor="red",
    opacity=0.2,
    layer="below",
    line_width=0
)

fig.update_layout(
    xaxis={'categoryorder': 'total descending'},
    legend_title_text='Fuel Source'
)

fig.show()


In [ ]:
df['date'] = pd.to_datetime(df['time']).dt.date

fig = px.box(
    df,
    x='date',                # Each box is a day
    y='dev_from_avg',            # Show the spread of TOTALMWh
    color='fuel_source_primary',  # Color by technology
    points=False,            # Show all points (optional)
    title='Spread of Generation by Technology for Each Day',
    labels={
        'TOTALMWh': 'Generation (MWh)',
        'date': 'Date',
        'fuel_source_primary': 'Technology'
    },
    height=500
)

# Add shaded rectangle for a date range
fig.add_vrect(
    x0="2018-12-27",
    x1="2019-02-10",
    fillcolor="red",
    opacity=0.2,
    layer="below",
    line_width=0
)

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Generation (MWh)',
    legend_title_text='Technology'
)

fig.show()

In [ ]:
djf_df = df.set_index('time').loc['2018-12-01':'2019-02-28'].reset_index()
djf_df['EHF_flag'] = djf_df['EHF_flag'].astype(bool)

In [ ]:

# Unique fuel types
fuel_types = djf_df['fuel_source_primary'].unique()

fig = go.Figure()
trace_labels = []

# Step 1: Add histogram and KDE curve per fuel type
for fuel in fuel_types:
    for label, flag in [('HW', True), ('non-HW', False)]:
        color = 'red' if label == 'HW' else 'blue'
        data = djf_df[(djf_df['fuel_source_primary'] == fuel) & (djf_df['EHF_flag'] == flag)]['TOTALMWh'].dropna()

        # Histogram
        fig.add_trace(
            go.Histogram(
                x=data,
                name=f'{fuel} - {label}',
                opacity=0.6,
                visible=True,  # default to visible for "All"
                histnorm='probability density',
                nbinsx=50,
                marker_color=color
            )
        )
        trace_labels.append((fuel, label))

        # KDE curve
        if len(data) > 1:
            kde = gaussian_kde(data)
            x_vals = np.linspace(data.min(), data.max(), 200)
            y_vals = kde(x_vals)

            fig.add_trace(
                go.Scatter(
                    x=x_vals,
                    y=y_vals,
                    mode='lines',
                    name=f'{fuel} - {label} KDE',
                    visible=True,
                    line=dict(color=color, dash='dash')
                )
            )
            trace_labels.append((fuel, f'{label}_KDE'))

# Step 2: Create dropdown buttons
dropdown_buttons = []

# All: show everything
dropdown_buttons.append(
    dict(
        label='All',
        method='update',
        args=[
            {"visible": [True] * len(trace_labels)},
            {"title": {"text": "Unit-Normalized Daily Energy Generation – All Fuel Types"}}
        ]
    )
)

# One button per fuel type
for fuel in fuel_types:
    visible = [(f == fuel) for f, _ in trace_labels]
    dropdown_buttons.append(
        dict(
            label=fuel,
            method='update',
            args=[
                {"visible": visible},
                {"title": {"text": f"Unit-Normalized Daily Energy Generation – {fuel} (HW vs non-HW + KDEs)"}}
            ]
        )
    )

# Step 3: Layout
fig.update_layout(
    title="Unit-Normalized Daily Energy Generation",
    xaxis_title="Generation (MWh)",
    yaxis_title="Density",
    barmode="overlay",
    updatemenus=[
        dict(
            buttons=dropdown_buttons,
            direction='down',
            x=1.05,
            xanchor='left',
            y=1.15,
            yanchor='top'
        )
    ]
)

fig.show()


In [ ]:
# fig = px.line(
#     ftype_df,
#     x='time',
#     y='dev_from_avg',
#     color='fuel_source_primary',
#     labels={
#         'time': 'Date',
#         'Clear Gap': 'Clear Gap (MWh)',
#         'fuel_source_primary': 'Fuel Source'
#     },
#     title='EHF by DUID'
# )

# # Add shaded heatwave area using secondary y-axis
# fig.add_trace(go.Scatter(
#     x=gens_percent.index,
#     y=gens_percent[1],
#     fill='tozeroy',
#     mode='none',
#     fillcolor='rgba(255, 100, 100, 0.2)',
#     name='% in Heatwave (shading)',
#     yaxis='y2',
#     hoverinfo='x+y'
# ))

# fig.add_trace(go.Scatter(
#     x=demand.index,
#     y=demand['TOTALDEMAND'],
#     mode='lines',
#     line=dict(color='red', dash='dash'),
#     fillcolor='rgba(255, 100, 100, 0.2)',
#     name='Demand (MWh)',
#     hoverinfo='x+y'
# ))

# # Update layout to define yaxis2
# fig.update_layout(
#     yaxis=dict(
#         title='Clear Gap (MWh)'
#     ),
#     yaxis2=dict(
#         title='% in Heatwave',
#         overlaying='y',
#         side='right',
#         ticksuffix='%',
#         range=[0, 100],  # adjust based on your data
#         showgrid=False
#     ),
#     xaxis_title='Date',
#     height=600,
#     legend_title='Fuel Source',
#         legend=dict(
#         x=1.2,  # Place legend to the right
#         y=1,
#         xanchor='left',
#         yanchor='top'
#     )
# )

# fig.show()

In [ ]:
def size(df,col,min_size = 3,max_size = 30):
    sizes = df[col].abs()
    if sizes.max() > 0:
        normalized_sizes = (sizes - sizes.min()) / (sizes.max() - sizes.min())
        df['marker_size'] = normalized_sizes * (max_size - min_size) + min_size
    else:
        df['marker_size'] = min_size
    return df

In [ ]:
splot_df = size(djf_df.copy(),'dev_from_avg')
splot_df['EHF_label'] = splot_df['EHF_flag'].astype(bool)
splot_df['EHF_label'] = pd.Categorical(
    splot_df['EHF_label'], categories=[False, True]
)

In [ ]:
splot_df['EHF_flag'] = splot_df['EHF_flag'].astype(int)  # Ensure integer type

fig = px.scatter_map(
    splot_df,
    lat='lat_jittered',
    lon='lon_jittered',
    animation_frame="date",
    hover_name='DUID',
    hover_data={
        'lat': False,
        'lon': False,
        'lat_jittered': False,
        'lon_jittered': False,
        'fuel_source_primary': True,
        'marker_size': False,
        'TOTALMWh': True,
        'dev_from_avg': True,
        'grouped_cum_avg':True,
        'EHF_flag':False
    },
    size='marker_size',
    color='EHF_flag',
    color_continuous_scale='Picnic',  # Reversed for "blue = 0, red = 1"
    range_color=(0, 1),                 # Ensure full 0–1 range is respected
    labels={
        'fuel_source_primary': 'Fuel Source',
        'grouped_cum_avg': 'Cumulative Average',
        'TOTALMWh': 'Daily Generation',
        'dev_from_avg':'Deviation from Average'
    },
    zoom=5,
    map_style='carto-positron'
)

fig.update_layout(
    coloraxis_colorbar=dict(
        title="EHF Event",
        tickvals=[0, 1],
        ticktext=["No Heatwave", "Heatwave"]
    ),
    title_text="Heatwave Events on Map"
)

fig.show()
fig.write_html("/g/data/ng72/ms5578/ID_HW_BARRA/data/output/NSW_18_19/b2_on_map.html")